# S08 toy — observability & replay, transparent

A pub-quiz host ("Trivia Night") runs a phased session against a **mock model** (a plain
Python function). You will instrument it with spans and generations, record every model
call to a JSONL **cassette**, replay the session offline — and then hunt the
nondeterminism that makes two runs of the same session differ. No network, no keys,
no cost.

**How to use:** run cells in order. Each experiment has an *attempt* cell (write your
prediction there before running anything) followed by a clearly marked *solution* cell.
The gap between prediction and result is the lesson.

## The system under observation

The host is a stateless-task client: every model call is one user message tagged
`TASK:...`, and the mock routes on the tag (a real deployment routes on the system
prompt; the mechanism is the same). One thing to notice in the response envelope:
`id` and `created` are **volatile** — real APIs stamp those per call. Content is the
contract; the envelope is per-call noise.

In [ ]:
\
import contextlib
import datetime
import difflib
import hashlib
import json
import random
import tempfile
import time
from pathlib import Path

BANK = {
    "science": [
        ("Which planet is home to the Great Red Spot?", "jupiter"),
        ("Which gas do plants absorb from the air?", "carbon dioxide"),
        ("What is H2O more commonly called?", "water"),
    ],
}

MOCK_CALLS = 0   # live-model invocations; replay must never move this counter

def mock_model(messages):
    """Stateless stand-in for POST /v1/chat/completions: one task-tagged request in,
    one API-shaped response out. Content is deterministic; the envelope is volatile,
    like the real thing."""
    global MOCK_CALLS
    MOCK_CALLS += 1
    head, _, player = messages[-1]["content"].partition(" PLAYER:")
    kv = dict(part.split(":", 1) for part in head.split())
    task, category = kv["TASK"], kv.get("CATEGORY", "science")
    if task == "welcome":
        content = f"Welcome to Trivia Night. Category: {category}. Three questions, no lifelines."
    elif task == "question":
        content = BANK[category][int(kv["N"]) - 1][0]
    elif task == "grade":
        key = BANK[category][int(kv["N"]) - 1][1]
        verdict = "CORRECT" if key in player.lower() else "WRONG"
        content = f"{verdict} — the answer was '{key}'."
    elif task == "scorecard":
        content = f"Final score {kv['SCORE']}. The bar tab remembers everything."
    else:
        raise ValueError(f"unknown task {task!r}")   # real APIs reject malformed input too
    prompt_tokens = sum(len(m["content"]) for m in messages) // 4
    completion_tokens = len(content) // 4
    return {
        "id": f"chatcmpl-{MOCK_CALLS:04d}",     # volatile: differs across live runs
        "created": int(time.time()),            # volatile: wall clock, like real APIs
        "choices": [{"message": {"role": "assistant", "content": content}}],
        "usage": {"prompt_tokens": prompt_tokens,
                  "completion_tokens": completion_tokens,
                  "total_tokens": prompt_tokens + completion_tokens},
    }

## The tracer

One `Tracer` instance == one **trace**. A `span` is a named, timed unit of work with a
parent; a `generation` is a span that also carries `input` / `output` / `usage`. The
tree — not any single span — is what makes "which phase burned the tokens?" an
arithmetic question.

The clock is *injected* (`TickClock`): real tracers call `time.monotonic()`; the toy
substitutes a deterministic tick so tree renders are reproducible. Hold that thought —
injected clocks come back in experiment 5.

In [ ]:
\
class TickClock:
    """Deterministic stand-in for the wall clock: +0.4 s per read."""
    def __init__(self): self.t = 0.0
    def __call__(self):
        self.t += 0.4
        return self.t

class Tracer:
    """Minimal trace tree: nested spans; generations carry usage."""
    def __init__(self, exporter=None, clock=None):
        self.clock = clock or TickClock()
        self.exporter = exporter or (lambda trace: None)
        self.roots, self._stack = [], []

    @contextlib.contextmanager
    def span(self, name, kind="span", **attrs):
        node = {"name": name, "kind": kind, "attrs": dict(attrs),
                "children": [], "t0": self.clock()}
        (self._stack[-1]["children"] if self._stack else self.roots).append(node)
        self._stack.append(node)
        try:
            yield node["attrs"]            # the caller writes output/usage here
        finally:
            self._stack.pop()
            node["duration_s"] = round(self.clock() - node["t0"], 3)

    def generation(self, name, **attrs):
        return self.span(name, kind="generation", **attrs)

    def export(self):
        """Ship the finished trace. May raise — the caller must catch (experiment 2)."""
        self.exporter({"spans": self.roots})

    def render(self):
        lines = []
        def walk(node, depth):
            extra = ""
            if node["kind"] == "generation" and "usage" in node["attrs"]:
                extra = f"  ({node['attrs']['usage']['total_tokens']} tok)"
            lines.append("  " * depth
                         + f"{node['name']} [{node['kind']}] {node['duration_s']:.1f}s{extra}")
            for child in node["children"]:
                walk(child, depth + 1)
        for root in self.roots:
            walk(root, 0)
        return "\n".join(lines)

class NullTracer:
    """Same interface, no work: instrumentation without `if tracer:` branches."""
    def span(self, name, kind="span", **attrs): return contextlib.nullcontext({})
    def generation(self, name, **attrs):        return contextlib.nullcontext({})
    def export(self): pass
    def render(self): return "(tracing disabled)"

NULL_TRACER = NullTracer()

## The host, instrumented

A span per phase (`setup`, `rounds`, `round-N`, `wrap-up`), a generation per model
call, and `export()` at the boundary — wrapped so telemetry can fail without taking
the quiz down (`fail_soft`). The player is **scripted** (S02's lesson: reproducibility
comes from the script, not from hoping a model repeats itself).

In [ ]:
\
PLAYER = ["Jupiter", "oxygen", "water"]   # scripted: right, wrong, right

def run_session(player, model, category="science", tracer=NULL_TRACER, fail_soft=True):
    """Trivia Night: setup -> 3 rounds -> wrap-up. Returns the rendered transcript."""
    transcript = []

    def llm(task, name):
        with tracer.generation(name, input=task) as g:
            body = model([{"role": "user", "content": task}])   # fresh list per call
            g["output"] = body["choices"][0]["message"]["content"]
            g["usage"] = body["usage"]
        return body["choices"][0]["message"]["content"]

    with tracer.span("session"):
        with tracer.span("setup"):
            transcript.append(f"[setup] {llm(f'TASK:welcome CATEGORY:{category}', 'welcome')}")
        with tracer.span("rounds"):
            score = 0
            for i, answer in enumerate(player, start=1):
                with tracer.span(f"round-{i}"):
                    q = llm(f"TASK:question CATEGORY:{category} N:{i}", f"q{i}")
                    verdict = llm(f"TASK:grade CATEGORY:{category} N:{i} PLAYER:{answer}",
                                  f"grade{i}")
                    score += verdict.startswith("CORRECT")
                    transcript.append(f"[q{i}] {q}")
                    transcript.append(f"[you] {answer}")
                    transcript.append(f"[host] {verdict}")
        with tracer.span("wrap-up"):
            closing = llm(f"TASK:scorecard SCORE:{score}/3", "scorecard")
            transcript.append(f"[final {score}/3] {closing}")

    if fail_soft:   # telemetry degrades; the product never does
        try:
            tracer.export()
        except Exception as exc:
            print(f"[telemetry disabled] export failed: {type(exc).__name__}: {exc}")
    else:
        tracer.export()
    return transcript

## Experiment 1 — trace anatomy

**Predict first:** sketch the tree before you render it — how many spans, how many
generations, what nests under `rounds`? Fill in your guesses, then check them against
the solution.

In [ ]:
# YOUR ATTEMPT — experiment 1
# Predicted tree (finish the sketch):
#   session
#     setup -> generation: welcome
#     ???
expected_generations = None   # your guess: how many generation spans?
expected_spans = None         # your guess: how many plain spans?

In [ ]:
# SOLUTION — experiment 1 (run after your attempt)
tracer = Tracer()
transcript = run_session(PLAYER, mock_model, tracer=tracer)
print(tracer.render())

def collect(node, kind):
    found = [node] if node["kind"] == kind else []
    for child in node["children"]:
        found += collect(child, kind)
    return found

gens  = [n for root in tracer.roots for n in collect(root, "generation")]
spans = [n for root in tracer.roots for n in collect(root, "span")]
toks  = sum(n["attrs"]["usage"]["total_tokens"] for n in gens)
print(f"\n{len(spans)} spans, {len(gens)} generations, {toks} total tokens")
print(f"your guess: {expected_generations} generations, {expected_spans} spans")
assert len(gens) == 8 and len(spans) == 7   # welcome + 3x(question+grade) + scorecard

## Experiment 2 — the exporter is down

The tracing backend is unreachable: `export()` raises `ConnectionError`.
**Predict first:** does the quiz still happen? Does the scorecard reach the player —
and what was *paid for* before the failure?

In [ ]:
# YOUR ATTEMPT — experiment 2
# Predictions:
#   with fail_soft=False the caller gets ... and the model calls ...
#   with fail_soft=True  the caller gets ... and the trace ...
# Which loss is worse: the traces, or the finished session?

In [ ]:
# SOLUTION — experiment 2 (run after your attempt)
def dead_exporter(trace):
    raise ConnectionError("tracing backend unreachable (simulated)")

before = MOCK_CALLS
try:
    run_session(PLAYER, mock_model,
                tracer=Tracer(exporter=dead_exporter), fail_soft=False)
    print("unguarded: session completed (unexpected)")
except ConnectionError as exc:
    print(f"unguarded: ConnectionError reached the caller — {exc}")
    print(f"unguarded: {MOCK_CALLS - before} model calls ran and were 'paid for';"
          " the finished transcript died with the exception")

transcript = run_session(PLAYER, mock_model, tracer=Tracer(exporter=dead_exporter))
print(f"guarded: session completed — {len(transcript)} transcript lines; last line:")
print(" ", transcript[-1])
assert len(transcript) == 11
print("telemetry degrades; the product never does")

## The cassette

`Recorder` wraps the model and appends `{request, response}` to a JSONL file.
`Replayer` answers from that file — **strictly**: the full request must equal the
**next unused** recorded request, in cassette order, or it raises `ReplayMismatch`.
Serving responses in order without matching would hide behavior drift; matching turns
the cassette into a contract — and because the match is positional, a program that
reorders two distinct calls trips it too.

Strict request matching and complete cassette consumption are **separate
invariants**: matching polices the calls that *arrive*; `assert_exhausted()` at
session end polices the calls that *should have arrived* — a regression that deletes
the final model call sends nothing bad to match, so only the second invariant
catches it.
(This is VCR's cassette idea, moved up one layer: from HTTP to the model function.)

In [ ]:
\
class ReplayMismatch(AssertionError):
    """A request reached the replayer that the next cassette entry does not match."""

class Recorder:
    """Model wrapper: forwards to the live model, appends each pair to JSONL."""
    def __init__(self, model, path):
        self.model, self.path = model, Path(path)
        self.path.write_text("")                      # start a fresh cassette

    def __call__(self, messages):
        response = self.model(messages)
        with self.path.open("a", encoding="utf-8") as fh:
            fh.write(json.dumps({"request": messages, "response": response}) + "\n")
        return response

class Replayer:
    """Cassette-backed model. Two SEPARATE invariants: strict matching — the FULL
    request must equal the NEXT unused entry, in cassette order, no match =>
    ReplayMismatch, never a guess; and exhaustion — assert_exhausted() at session
    end, so a regression that DELETES a call (nothing bad arrives to match) fails too."""
    def __init__(self, path):
        self.entries = [json.loads(line) for line in
                        Path(path).read_text(encoding="utf-8").splitlines()]
        self._next = 0

    def __call__(self, messages):
        if self._next < len(self.entries):
            entry = self.entries[self._next]
            if entry["request"] == messages:
                self._next += 1
                return entry["response"]       # recorded content, parsed back from JSON
        raise ReplayMismatch("next cassette entry does not match request: "
                             + json.dumps(messages))

    def assert_exhausted(self):
        """Session-end invariant: every recorded call must have been consumed."""
        unused = len(self.entries) - self._next
        if unused:
            raise ReplayMismatch(
                f"{unused} cassette {'entry' if unused == 1 else 'entries'} never used — "
                "the session made fewer calls than the recording")

## Experiment 3 — record, then replay

**Predict first:** how many times will the live model fire during replay? Will the two
transcripts be byte-identical — and what could possibly differ?

In [ ]:
# YOUR ATTEMPT — experiment 3
# Predictions:
#   MOCK_CALLS after the replay = ?
#   live == replayed ?   (if not, which lines differ, and why?)

In [ ]:
# SOLUTION — experiment 3 (run after your attempt)
tmp = tempfile.TemporaryDirectory()
cassette = Path(tmp.name) / "trivia-night.jsonl"

MOCK_CALLS = 0
live = run_session(PLAYER, Recorder(mock_model, cassette))
lines = cassette.read_text(encoding="utf-8").splitlines()
print(f"live run: {MOCK_CALLS} model calls -> cassette of {len(lines)} JSONL lines")
print("line 1, truncated:", lines[0][:110], "...\n")

MOCK_CALLS = 0
replayer = Replayer(cassette)
replayed = run_session(PLAYER, replayer)
replayer.assert_exhausted()   # session invariant: nothing recorded was left unserved
sha = lambda t: hashlib.sha256("\n".join(t).encode()).hexdigest()[:16]
print(f"replay:   {MOCK_CALLS} model calls (must be 0 — offline by construction)")
print(f"transcript sha256  live={sha(live)}  replay={sha(replayed)}")
assert MOCK_CALLS == 0, "replay touched the live model"
assert live == replayed, "replay diverged from the recording"
print("byte-identical transcript, zero model calls, cassette fully consumed —")
print("matching checked what ARRIVED; exhaustion checked that everything RECORDED arrived")

## Experiment 4 — replay as tripwire

The player now answers "Mars" to question 1, but the cassette still holds the Jupiter
session. **Predict first:** where does replay break — at the first call, at the changed
call, or never? And what would a serve-in-order replayer *without* matching have done
with the changed request?

Then the quieter regression: the host loses its final model call entirely.
**Predict first:** does strict request matching fire on a call that never happens —
and which invariant does?

In [ ]:
# YOUR ATTEMPT — experiment 4
# Predictions:
#   the mismatch surfaces at generation #__ because ...
#   a sequential replayer would have ...

In [ ]:
# SOLUTION — experiment 4 (run after your attempt)
different_player = ["Mars", "oxygen", "water"]   # Q1 answer changed; cassette is stale
try:
    run_session(different_player, Replayer(cassette))
    print("replay completed (unexpected)")
except ReplayMismatch as exc:
    print("the replayer refused to continue:")
    print(" ", str(exc)[:150])

print()
print("A strict cassette is a contract: behavior drift is a loud mismatch, not a")
print("silently wrong transcript. Serve-in-order WITHOUT matching would have served")
print("the recorded 'CORRECT' verdict for the Mars answer — wrong, and invisible.")

# second failure mode: a regression DELETES the final model call (no scorecard).
# Nothing bad ever arrives, so strict matching alone replays the prefix clean...
regressed = Replayer(cassette)
recorded = [json.loads(line) for line in cassette.read_text(encoding="utf-8").splitlines()]
for entry in recorded[:-1]:                # the session ends one call early
    regressed(entry["request"])            # every request matches, in order — no mismatch
print(f"\ndeleted-call regression: {len(recorded) - 1}/{len(recorded)} recorded calls served,")
print("strict matching stayed silent (every arriving request matched) — then:")
try:
    regressed.assert_exhausted()
    print("exhaustion check passed (unexpected)")
except ReplayMismatch as exc:
    print(f"assert_exhausted() raised: {exc}")

## Experiment 5 — the hunt (find the planted nondeterminism)

A teammate ships a "harmless" PR: a dated header, livelier praise. Now two live runs of
the same session differ. **Predict first:** is the nondeterminism in the model or in
the host — and which experiment separates the two *without reading the code*? List your
suspects, then run the solution.

In [ ]:
# YOUR ATTEMPT — experiment 5
# Model or client? My guess, with reasoning: ...
# Suspects:
#   1) ...
#   2) ...

In [ ]:
# SOLUTION part 1 — experiment 5: reproduce, then separate model from client

# --- the teammate's PR: v2 of the host. Two of these lines are defects. ---
PRAISE = ["Nice!", "Sharp!", "Clean hit!", "The crowd goes mild."]

def run_session_v2(player, model, category="science"):
    transcript = [f"TRIVIA NIGHT — {datetime.datetime.now().isoformat()}"]  # PR: dated header
    def llm(task):
        body = model([{"role": "user", "content": task}])
        return body["choices"][0]["message"]["content"]
    transcript.append(f"[setup] {llm(f'TASK:welcome CATEGORY:{category}')}")
    score = 0
    for i, answer in enumerate(player, start=1):
        q = llm(f"TASK:question CATEGORY:{category} N:{i}")
        verdict = llm(f"TASK:grade CATEGORY:{category} N:{i} PLAYER:{answer}")
        if verdict.startswith("CORRECT"):
            verdict += " " + random.choice(PRAISE)                          # PR: livelier praise
            score += 1
        transcript += [f"[q{i}] {q}", f"[you] {answer}", f"[host] {verdict}"]
    transcript.append(f"[final {score}/3] {llm(f'TASK:scorecard SCORE:{score}/3')}")
    return transcript

live1 = run_session_v2(PLAYER, mock_model)
live2 = run_session_v2(PLAYER, mock_model)
print("two live runs identical?", live1 == live2)
print("\n".join(difflib.unified_diff(live1, live2, "live-1", "live-2", lineterm="")))

# the separating experiment: replay ONE cassette twice
cassette_v2 = Path(tmp.name) / "trivia-v2.jsonl"
run_session_v2(PLAYER, Recorder(mock_model, cassette_v2))
rep1 = run_session_v2(PLAYER, Replayer(cassette_v2))
rep2 = run_session_v2(PLAYER, Replayer(cassette_v2))
print("\ntwo replays of ONE cassette identical?", rep1 == rep2)
print("\n".join(difflib.unified_diff(rep1, rep2, "replay-1", "replay-2", lineterm="")))
print("""
The recorded responses are frozen content (parsed JSON) — replay CANNOT add variance.
So the nondeterminism is in the host, and the diff names the two suspects:
the header timestamp (datetime.now) and the praise draw (random.choice).""")

### The fix: inject both sources

Determinism is not a property you hope for; it is a property you *wire in*. Pass the
clock and the RNG as parameters — and create **fresh seeded instances per run**: one
shared `Random(7)` reproduces across processes but drifts between two runs inside one
process.

In [ ]:
# SOLUTION part 2 — experiment 5: inject, seed, prove

def run_session_v3(player, model, category="science", *, clock, rng):
    transcript = [f"TRIVIA NIGHT — {clock()}"]              # injected clock
    def llm(task):
        body = model([{"role": "user", "content": task}])
        return body["choices"][0]["message"]["content"]
    transcript.append(f"[setup] {llm(f'TASK:welcome CATEGORY:{category}')}")
    score = 0
    for i, answer in enumerate(player, start=1):
        q = llm(f"TASK:question CATEGORY:{category} N:{i}")
        verdict = llm(f"TASK:grade CATEGORY:{category} N:{i} PLAYER:{answer}")
        if verdict.startswith("CORRECT"):
            verdict += " " + rng.choice(PRAISE)             # injected, seeded RNG
            score += 1
        transcript += [f"[q{i}] {q}", f"[you] {answer}", f"[host] {verdict}"]
    transcript.append(f"[final {score}/3] {llm(f'TASK:scorecard SCORE:{score}/3')}")
    return transcript

def fresh_deps():
    return {"clock": lambda: "2026-08-13T21:00:00", "rng": random.Random(7)}

a = run_session_v3(PLAYER, mock_model, **fresh_deps())
b = run_session_v3(PLAYER, mock_model, **fresh_deps())
replayer_v3 = Replayer(cassette_v2)
c = run_session_v3(PLAYER, replayer_v3, **fresh_deps())
replayer_v3.assert_exhausted()   # v3 must re-request the whole recording, not a prefix
print("live == live:", a == b, " | live == replay:", a == c)
assert a == b == c
print("byte-identical everywhere: determinism is a property of the whole pipeline")

## What transfers to the real build

- `Tracer.span` / `Tracer.generation` → your tracing SDK's span and generation calls —
  same tree: a span per phase, a generation per model call, usage on the generation.
- `export()` behind `fail_soft` → the try/except at the telemetry boundary: a dead
  observability backend must never fail a session. SDK version drift makes this a
  *when*, not an *if*.
- `Recorder` / `Replayer` + JSONL → the record/replay flag: persist every
  request/response pair; strict request matching plus an end-of-session exhaustion
  check turns behavior drift — changed AND deleted calls — into a loud mismatch
  instead of a wrong transcript.
- the two-replays diagnostic → how you prove nondeterminism is client-side *before*
  hunting it.
- injected `clock` / `rng`, fresh per run → how fixtures stay byte-identical.
- what the toy doesn't have: async spans and cross-process context propagation,
  redaction (traces carry transcripts — telemetry inherits the privacy boundary), real
  exporter protocols (OTLP), and SDK churn. Those are the real build's job.

Now do the real build in your own project. You type it.